In [18]:
#!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [19]:
# %cd capstone_project_GroupA
# !git checkout main

In [20]:
# %cd src

In [21]:
# if you put this into another folder within src, u need below.
# import sys, os
# sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import glob
from NSWData.NSWDataLoader import *
from ModelFiles.ModelPlots import *
nsw_data_loader = NSWDataLoader()

Found NSW data path: C:\Users\kyim1\Desktop\capstone_project_GroupA\data\NSW


In [22]:
sarimax_folder = os.path.join(nsw_data_loader.output_dir, 'SARIMAX_results')
metrics_files = glob.glob(os.path.join(sarimax_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
sarimax_results_df = pd.concat(dfs, ignore_index=True)

gb_folder = os.path.join(nsw_data_loader.output_dir, 'gradient_boosting_results')
metrics_files = glob.glob(os.path.join(gb_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
gb_results_df = pd.concat(dfs, ignore_index=True)

lstm_folder = os.path.join(nsw_data_loader.output_dir, 'LSTM_results')
metrics_files = glob.glob(os.path.join(lstm_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
lstm_results_df = pd.concat(dfs, ignore_index=True)

patchtst_folder = os.path.join(nsw_data_loader.output_dir, 'PATCHTST_results')
metrics_files = glob.glob(os.path.join(patchtst_folder, "*_test_metrics.csv"))
dfs = [pd.read_csv(f) for f in metrics_files]
patchtst_results_df = pd.concat(dfs, ignore_index=True)

In [23]:
sarimax_results_df['model'] = 'SARIMAX'
sarimax_results_df['seed'] = '31415'
sarimax_results_df = sarimax_results_df[['model', 'seed', 'horizon','rmse','mae']]
gb_results_df['model'] = 'Gradient Boosting'
gb_results_df = gb_results_df[['model', 'seed', 'horizon','rmse','mae']]
lstm_results_df['model'] = 'LSTM'
lstm_results_df = lstm_results_df[['model', 'seed', 'horizon','rmse','mae']]
patchtst_results_df['model'] = 'PATCHTST'
patchtst_results_df = patchtst_results_df[['model', 'seed', 'horizon','rmse','mae']]
all_results_df = pd.concat([sarimax_results_df, gb_results_df, lstm_results_df, patchtst_results_df], ignore_index=True)

In [24]:
all_results_df = all_results_df.groupby(['model', 'horizon'])[['rmse','mae']].agg(['mean', 'std', 'size']).reset_index()
t_crit = all_results_df[('rmse', 'size')].apply(lambda n: stats.t.ppf(0.975, df=n - 1))
se = all_results_df[('rmse', 'std')] / all_results_df[('rmse', 'size')] ** 0.5
all_results_df[('rmse', 'CI_lower')] = all_results_df[('rmse', 'mean')] - t_crit * se
all_results_df[('rmse', 'CI_upper')] = all_results_df[('rmse', 'mean')] + t_crit * se
se = all_results_df[('mae', 'std')] / all_results_df[('mae', 'size')] ** 0.5
all_results_df[('mae', 'CI_lower')] = all_results_df[('mae', 'mean')] - t_crit * se
all_results_df[('mae', 'CI_upper')] = all_results_df[('mae', 'mean')] + t_crit * se

In [25]:
all_results_df

model horizon        rmse                         mae  \
                                     mean        std size        mean   
0   Gradient Boosting      48  536.341414   0.009260    6  380.493724   
1   Gradient Boosting     336  655.670469   0.123945    6  471.720643   
2   Gradient Boosting     720  690.824622   0.169271    6  498.168359   
3                LSTM      48  564.984483  20.864307    6  402.084856   
4                LSTM     336  790.935147  17.104175    6  574.378708   
5                LSTM     720  858.991732  13.236071    6  635.970815   
6            PATCHTST      48  468.070516   3.867112    6  308.625024   
7            PATCHTST     336  637.350057  14.995018    6  445.230645   
8            PATCHTST     720  694.992606  19.408241    6  496.406005   
9             SARIMAX      48  544.645728        NaN    1  367.250725   
10            SARIMAX     336  650.793036        NaN    1  454.654965   
11            SARIMAX     720  667.581898        NaN    1  471.884655   

                          rmse                     mae              
          std size    CI_lower    CI_upper    CI_lower    CI_upper  
0    0.012602    6  536.331696  536.351132  380.480499  380.506949  
1    0.164631    6  655.540397  655.800541  471.547873  471.893412  
2    0.168273    6  690.646983  691.002261  497.991767  498.344951  
3   13.513413    6  543.088735  586.880231  387.903399  416.266313  
4   16.850085    6  772.985416  808.884879  556.695628  592.061788  
5   10.637197    6  845.101327  872.882137  624.807761  647.133869  
6    2.717366    6  464.012231  472.128802  305.773324  311.476725  
7   14.990098    6  621.613751  653.086364  429.499502  460.961789  
8   18.384209    6  674.624906  715.360305  477.112960  515.699049  
9         NaN    1         NaN         NaN         NaN         NaN  
10        NaN    1         NaN         NaN         NaN         NaN  
11        NaN    1         NaN         NaN         NaN         NaN

In [27]:
all_results_df[[('rmse','mean'),('rmse','std'),('rmse','CI_lower'),('rmse','CI_upper'),('mae','mean'),('mae','std'),('mae','CI_lower'),('mae','CI_upper')]] = all_results_df[[('rmse','mean'),('rmse','std'),('rmse','CI_lower'),('rmse','CI_upper'),('mae','mean'),('mae','std'),('mae','CI_lower'),('mae','CI_upper')]].round(2)
all_results_df.to_csv(os.path.join(nsw_data_loader.output_dir, 'model_comparison_results.csv'), index=False)
print(all_results_df)

                model horizon    rmse                 mae              \
                                 mean    std size    mean    std size   
0   Gradient Boosting      48  536.34   0.01    6  380.49   0.01    6   
1   Gradient Boosting     336  655.67   0.12    6  471.72   0.16    6   
2   Gradient Boosting     720  690.82   0.17    6  498.17   0.17    6   
3                LSTM      48  564.98  20.86    6  402.08  13.51    6   
4                LSTM     336  790.94  17.10    6  574.38  16.85    6   
5                LSTM     720  858.99  13.24    6  635.97  10.64    6   
6            PATCHTST      48  468.07   3.87    6  308.63   2.72    6   
7            PATCHTST     336  637.35  15.00    6  445.23  14.99    6   
8            PATCHTST     720  694.99  19.41    6  496.41  18.38    6   
9             SARIMAX      48  544.65    NaN    1  367.25    NaN    1   
10            SARIMAX     336  650.79    NaN    1  454.65    NaN    1   
11            SARIMAX     720  667.58    NaN    1  